# Setup for exercise B

In [ ]:
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
img_name = input("Image name: ")
read_img = cv.imread(f"{img_name}.jpg", cv.IMREAD_GRAYSCALE)
if read_img is None:
    raise FileNotFoundError(f"Image \"{img_name}.jpg\" not found.")
img: np.ndarray = read_img
img_width = img.shape[1]
img_height = img.shape[0]

# B1

In [ ]:
def smoothing(image: np.ndarray, type: str, ksize: int):
    k = (ksize, ksize)
    if type == "average":
        return cv.blur(image, k)
    if type == "median":
        return cv.medianBlur(image.astype(np.uint8), ksize)
    if type == "gaussian":
        return cv.GaussianBlur(image, k, sigmaX=0, sigmaY=0)
    raise ValueError(f"Unsupported filter type: {type}")

filters = ["average", "median", "gaussian"]
kernel_sizes = [3, 5, 9]


In [ ]:
plt.figure(figsize=(12, 16))
plt.subplot(4, 3, (1, 3))
plt.title("Original")
plt.imshow(img, cmap="gray")
plt.axis("off")

idx = 4

for f in filters:
    for s in kernel_sizes:
        img_processed = smoothing(img, f, s)
        plt.subplot(4, 3, idx)
        plt.title(f"{f} filter, kernel size {s}x{s}")
        plt.imshow(img_processed, cmap="gray")
        plt.axis("off")
        idx += 1
plt.tight_layout()
plt.show()


# B2

In [ ]:
def add_gaussian_noise(image: np.ndarray, mean: float = 0, stddev: float = 30):
    noise = np.random.normal(loc=mean, scale=stddev, size=image.shape).astype(np.float32)
    output = np.clip(image + noise, 0, 255)
    return output
def add_salt_and_pepper_noise(image: np.ndarray, prob: float = 0.1): # typehinting + default value!
    output = image.copy()
    # Salt
    amt = np.ceil(prob * image.size * 0.5).astype(int) # the 0.5 is for half salt, half pepper
    salt_coords = [np.random.randint(0, i, amt) for i in image.shape]
    pepper_coords = [np.random.randint(0, i, amt) for i in image.shape]
    output[salt_coords[0], salt_coords[1]] = 255
    output[pepper_coords[0], pepper_coords[1]] = 0
    return output
def add_periodic_noise(image: np.ndarray, freq: float = 0.01, amp: float = 20):
    rows, cols = image.shape
    # Need meshgrid because sinusoid repeats in only 1 direction but np.sin needs both dimensions
    x = np.arange(cols)
    y = np.arange(rows)
    xx, yy = np.meshgrid(x, y)
    sinusoid = amp * np.sin(2 * np.pi * freq * xx)
    img_float = image.astype(np.float32)
    output = np.clip(img_float + sinusoid, 0, 255)
    return output


In [ ]:
img_gaussian = add_gaussian_noise(img, 0, 30)
img_sp = add_salt_and_pepper_noise(img, 0.1)
img_periodic = add_periodic_noise(img, 0.01, 20)


In [ ]:
images: list[tuple[str, np.ndarray]] = [("Original", img),
("Gaussian noise", img_gaussian),
("Salt and pepper noise", img_sp),
("Periodic noise", img_periodic)]

plt.figure(figsize=(18, 8))
for idx, (name, image) in enumerate(images, start=1):
    plt.subplot(2, 4, idx)
    plt.title(name)
    plt.imshow(image, cmap="gray")
    plt.axis("off")
    plt.subplot(2, 4, idx + 4)
    hist_color = "steelblue" if name.lower() == "original" else "red"
    plt.hist(image.ravel(), bins=256, range=(0, 255), color=hist_color)
    plt.ylim(0, 5000) # arbitrary upper limit for y-range because proper implementation of max detection is too long


| Noise type | Effect on image | Effect on histogram |
|:-|:-|:-|
| Gaussian | Smooth layer of noise | Gentler graph, no peaks (except for 0 and 255 due to np.clip) |
| Salt-and-pepper | Coarse layer of noise | Mostly the same as original, but with 2 tall peaks at 0 and 255 |
| Periodic | Wave pattern over image | Somewhat visible sinusoidal pattern over the graph |

# B3

In [ ]:
temp = input("""Which type of noisy image to process?\n
1 - Gaussian\n
2 - Salt-and-pepper\n
3 - Periodic\n""")
noisy_image: np.ndarray = images[int(temp)][1]

plt.figure(figsize=(12, 16))
plt.subplot(4, 3, (1, 3))
plt.title("Noisy image")
plt.imshow(noisy_image, cmap="gray")
plt.axis("off")

idx = 4
for f in filters:
    for s in kernel_sizes:
        img_processed = smoothing(noisy_image, f, s)
        plt.subplot(4, 3, idx)
        plt.title(f"{f} filter, kernel size {s}x{s}")
        plt.imshow(img_processed, cmap="gray")
        plt.axis("off")
        idx += 1
plt.tight_layout()
plt.show()

| Noise type | Best filter | Best kernel size | Explanation |
|:-|:-|:-|:-|
| Gaussian | Gaussian | 5x5 | Removes noise without blurring image too much, like with an average filter or higher kernel size |
| Salt-and-pepper | Median | 3x3 | Removes large peaks (e.g. 0 and 255) without blurring image too much |
| Periodic | :( | :( | Spatial filters cannot effectively remove periodic noise; a frequency domain filter would have performed better |

# B4

In [ ]:
img_laplacian = cv.Laplacian(src=img.astype(np.float32), ddepth=-1, ksize=3)
img_sharpened = img.astype(np.float32) - img_laplacian
img_laplacian = np.clip(img_laplacian, 0, 255)
img_sharpened = np.clip(img_sharpened, 0, 255)

images: list[tuple[str, np.ndarray]] = [("Original", img),
("Laplacian", img_laplacian),
("Sharpened", img_sharpened)]

plt.figure(figsize=(15, 6))
for idx, (name, image) in enumerate(images, start=1):
    plt.subplot(1, 3, idx)
    plt.title(name)
    plt.imshow(image, cmap="gray")
    plt.axis("off")


The edges of the image are crisper. The Laplacian filter highlights points with rapid changes in gray level, and the sharpened image is the difference between the Laplacian and the original images.

# B5

In [ ]:
fft_img = np.fft.fftshift(np.fft.fft2(img))
# Exaggerate period and amplitude to be clearer on magnitude spectrum
periodic_noise_freq = 0.2
periodic_noise_amp = 128
img_periodic = add_periodic_noise(img, periodic_noise_freq, periodic_noise_amp)
fft_img_periodic = np.fft.fftshift(np.fft.fft2(img_periodic))
magnitude_original = 20 * np.log(np.abs(fft_img) + 1)  # + 1 to avoid error for log(0)
magnitude_periodic = 20 * np.log(np.abs(fft_img_periodic) + 1)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(magnitude_original, cmap='gray')
plt.title('Original')
plt.axis('off')
# Note: harmonics may appear due to sine wave being [np.]clipped in add_periodic_noise()
plt.subplot(1, 2, 2)
plt.imshow(magnitude_periodic, cmap='gray')
plt.title('With periodic noise')
plt.axis('off')
plt.tight_layout()
plt.show()


# B6

A notch filter will be used, since periodic noise usually contains very few frequency components. Our clipped sine wave has 4 (evident by the 9 bright points including DC spaced equally on the x-axis), but a pure sine wave would have only 1.

In [ ]:
def notch_filter(noisy_fft, freq, width, height, harmonic_count=5, notch_radius=1):
    """Apply a notch filter to an *already shifted* FFT, and returns a shifted FFT with periodic noise filtered out.
    Inverse shift the output before computing the inverse FFT.
    Notching out a single frequency component may not be enough to completely filter out the noise, so the notch radius can be customized."""
    # Create a mask: 1 everywhere, 0 at noise frequencies
    mask = np.ones_like(noisy_fft)
    x_center, y_center = width // 2, height // 2
    for n in range(1, harmonic_count + 1):
        # Processing both positive-x and negative-x frequencies (since they are pairs symmetrical about the center)
        for sign in [-1, 1]:
            x_offset = int(sign * n * freq * width)
            y_range = slice(max(0, y_center - 1), min(height, y_center + 2)) # notch_radius not applied to y_range since noise only oscillates along x-axis
            x_range = slice(max(0, x_center + x_offset - notch_radius), min(width, x_center + x_offset + notch_radius + 1))
            mask[y_range, x_range] = 0
    # Apply the mask to the original FFT
    filtered_fft = noisy_fft * mask
    return filtered_fft

fft_img_filtered = notch_filter(fft_img_periodic, freq=periodic_noise_freq, width=img_width, height=img_height, harmonic_count=2, notch_radius=10)
img_filtered = np.fft.ifft2(np.fft.ifftshift(fft_img_filtered)).real
magnitude_filtered = 20 * np.log(np.abs(np.fft.fftshift(np.fft.fft2(img_filtered))) + 1)

plt.figure(figsize=(12, 10))
plt.subplot(2, 2, 1)
plt.title('Image with periodic noise')
plt.imshow(img_periodic, cmap='gray')
plt.axis('off')
plt.subplot(2, 2, 2)
plt.title('Filtered image')
plt.imshow(img_filtered, cmap='gray')
plt.axis('off')
plt.subplot(2, 2, 3)
plt.imshow(magnitude_periodic, cmap='gray')
plt.axis('off')
plt.subplot(2, 2, 4)
plt.imshow(magnitude_filtered, cmap='gray')
plt.axis('off')
plt.tight_layout()
plt.show()